In [1]:
import pandas as pd
import numpy as np
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

# Download the VADER lexicon
nltk.download('vader_lexicon')

# Path to the filtered reviews from Notebook 2
reviews_path = 'data/reviews_filtered.csv'

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\rjana\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


We want the "compound" score, which is a metric that calculates the overall sentiment of a text from -1 (extremely negative) to +1 (extremely positive).

In [2]:
# Defining the Sentiment Function
analyzer = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    if pd.isna(text) or text == "":
        return 0.0
    # Returns the compound score directly
    return analyzer.polarity_scores(str(text))['compound']

Running this on millions of rows can be slow. We use .apply() but monitor progress.

In [3]:
chunk_size = 25000  # Smaller chunks to be extra safe
processed_chunks = []

print("Starting deep-memory sentiment processing...")

# We add:
# engine='python' -> slower but more stable for 'out of memory' tokenization
# on_bad_lines='skip' -> skips lines that might be corrupted or too long
reader = pd.read_csv(
    reviews_path, 
    chunksize=chunk_size, 
    engine='python', 
    on_bad_lines='skip'
)

try:
    for i, chunk in enumerate(reader):
        # Process sentiment
        chunk['sentiment_score'] = chunk['review_text'].apply(get_vader_sentiment)
        
        # Keep only the essential columns and drop the heavy text immediately
        chunk_slim = chunk[['book_id', 'sentiment_score', 'rating']].copy()
        processed_chunks.append(chunk_slim)
        
        if (i + 1) % 10 == 0:
            print(f"Processed {(i + 1) * chunk_size:,} reviews...")

except Exception as e:
    print(f"An error occurred during processing: {e}")

# Combine results
if processed_chunks:
    df_scores = pd.concat(processed_chunks, ignore_index=True)
    print(f"Successfully processed {len(df_scores):,} reviews.")
else:
    print("No data was processed.")

Starting deep-memory sentiment processing...
Processed 250,000 reviews...
Processed 500,000 reviews...
Processed 750,000 reviews...
Processed 1,000,000 reviews...
Processed 1,250,000 reviews...
Processed 1,500,000 reviews...
Processed 1,750,000 reviews...
Processed 2,000,000 reviews...
Processed 2,250,000 reviews...
Processed 2,500,000 reviews...
Processed 2,750,000 reviews...
Processed 3,000,000 reviews...
Processed 3,250,000 reviews...
Processed 3,500,000 reviews...
Processed 3,750,000 reviews...
Processed 4,000,000 reviews...
Processed 4,250,000 reviews...
Processed 4,500,000 reviews...
Processed 4,750,000 reviews...
Processed 5,000,000 reviews...
Processed 5,250,000 reviews...
Processed 5,500,000 reviews...
Processed 5,750,000 reviews...
Processed 6,000,000 reviews...
Processed 6,250,000 reviews...
Processed 6,500,000 reviews...
Successfully processed 6,626,356 reviews.


In [4]:
# The "Safe Aggregation" Cell
# 1. Aggregate
print("Aggregating 6.6M reviews to book level...")
sentiment_features = df_scores.groupby('book_id').agg({
    'sentiment_score': ['mean', 'std', 'count'],
    'rating': ['mean', 'std']
}).reset_index()

# 2. Flatten Columns
sentiment_features.columns = [
    'book_id', 'avg_sentiment', 'sentiment_std', 'review_count', 'avg_rating', 'rating_std'
]
sentiment_features = sentiment_features.fillna(0)

# 3. SAVE SAFETY CHECKPOINT (Crucial!)
sentiment_features.to_csv('data/temp_sentiment_features.csv', index=False)
print(f"Features created and SAVED to disk for {len(sentiment_features):,} books.")

# 4. PURGE MEMORY
del df_scores
import gc
gc.collect()
print("Giant review dataset cleared from RAM. You are safe now!")

Aggregating 6.6M reviews to book level...
Features created and SAVED to disk for 421,900 books.
Giant review dataset cleared from RAM. You are safe now!


In [5]:
# 1. Load the labeled books from Notebook 2
# We ensure we load 'author_reputation' which we just engineered
df_books = pd.read_csv('data/books_labeled.csv')

# 2. Merge with the sentiment features
# Using 'left' join ensures we keep 'author_reputation' and all other metadata
df_final = pd.merge(df_books, sentiment_features, on='book_id', how='left')

# 3. Fill missing values
# Important: If any book had 0 reviews, set sentiment/count to 0
cols_to_fill = ['avg_sentiment', 'sentiment_std', 'review_count', 'avg_rating', 'rating_std']
df_final[cols_to_fill] = df_final[cols_to_fill].fillna(0)

# 4. Final Save for Modeling
df_final.to_csv('data/final_modeling_data.csv', index=False)

print(f"Notebook 3 complete!")
print(f"Master modeling dataset saved with {len(df_final):,} books.")
print(f"Feature Check: 'author_reputation' present: {'author_reputation' in df_final.columns}")
print(f"Total Bestsellers identified: {df_final['bestseller'].sum()}")

C:\Users\rjana\AppData\Local\Temp\ipykernel_11956\3862841290.py:3: DtypeWarning: Columns (0: author_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df_books = pd.read_csv('data/books_labeled.csv')


Notebook 3 complete!
Master modeling dataset saved with 458,516 books.
Feature Check: 'author_reputation' present: True
Total Bestsellers identified: 28070
